<a href="https://colab.research.google.com/github/marcinwolter/MachineLearning-KISD-2026/blob/main/lecture8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>




#**<font color = "red">Introduction to machine learning</font>**

**Lecture 8**

##**<font color = "green">Large Language Models (LLM)</font>**
##**<font color = "green">Mixture Density Network (MDN)</font>**


*21 April 2026*


---

*Marcin Wolter, IFJ PAN*

*e-mail: marcin.wolter@ifj.edu.pl*


---
</center>

#<font color='green'>**Program for today:**



* ###  <font color='red'>Large language Models (LLM) - how is ChatGPT working?
* ###  <font color='red'>Mixture Density Network (MDN) - about network returning the probability distribution.

<br>


**As always all slides are here:**

*https://github.com/marcinwolter/MachineLearning-KISD-2026*

<br>




## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

np.random.seed(42)
plt.rcParams.update({
    'font.size': 12,
    'figure.figsize': (10, 5),
    'axes.spines.top': False,
    'axes.spines.right': False
})
print("Ready.")

<center>

---
---

# 🟢 Part 1: Why Did We Need a New Architecture?

---
---
</center>

## 1.1  The problem with sequences

Images have a fixed grid structure — that is why CNNs work so well on them. Text is different: it is a **variable-length sequence** where the meaning of each element depends on context that can be arbitrarily far away.

Our **running example** for this lecture:

> *"The trophy didn't fit in the suitcase because **it** was too big."*

What does *"it"* refer to — the **trophy** or the **suitcase**? A human resolves this by attending to both words simultaneously and using semantic reasoning. Early neural networks could not do this efficiently.

## 1.2  Recurrent Neural Networks (RNNs)

RNNs process sequences one token at a time, maintaining a hidden state $h_t$:

$$h_t = f(h_{t-1},\, x_t)$$

**Problems:**
- **Vanishing gradients** — gradients shrink exponentially over long sequences; early tokens are effectively forgotten
- **Sequential computation** — step $t$ cannot start until step $t-1$ finishes → cannot be parallelised on GPUs
- **Fixed-size bottleneck** — the entire sentence is compressed into one hidden vector $h_T$, regardless of sentence length

LSTMs and GRUs mitigate the gradient problem but do not solve the bottleneck or the parallelism issue.

*For RNN's and time-series forecasting see for example:*
https://deeplearningwithpython.io/chapters/chapter13_timeseries-forecasting/

## 1.3  The key idea: attention

Instead of compressing the whole sequence into one vector, what if every output position could **directly look at every input position**?

> *"Attention is All You Need"* — Vaswani et al., NeurIPS 2017

This paper replaced recurrence entirely and became the foundation of every modern LLM.

*For large language models see for example:* https://deeplearningwithpython.io/chapters/chapter15_language-models-and-the-transformer/

In [ ]:
# Diagram: RNN vs Attention information flow on the running example
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

tokens_ex = ["The", "trophy", "didn't", "fit", "because", "it"]
n = len(tokens_ex)

# --- RNN panel ---
ax = axes[0]
ax.set_title("RNN: Sequential information flow", fontsize=12, fontweight='bold')
for i, tok in enumerate(tokens_ex):
    ax.add_patch(plt.Circle((i, 0.5), 0.32, color='#4C72B0', zorder=3))
    ax.text(i, 0.5, tok, ha='center', va='center', color='white', fontsize=8.5, fontweight='bold')
    if i < n - 1:
        ax.annotate("", xy=(i+1-0.33, 0.5), xytext=(i+0.33, 0.5),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.8))
ax.annotate("", xy=(5-0.33, 0.15), xytext=(1+0.33, 0.15),
            arrowprops=dict(arrowstyle='->', color='crimson', lw=1.5, linestyle='dashed',
                            connectionstyle='arc3,rad=-0.35'))
ax.text(3, -0.22, '"trophy" info fades over 4 steps', ha='center', color='crimson', fontsize=9)
ax.set_xlim(-0.7, n-0.3); ax.set_ylim(-0.4, 1.1); ax.axis('off')

# --- Attention panel ---
ax = axes[1]
ax.set_title("Attention: Direct access to all positions", fontsize=12, fontweight='bold')
for i, tok in enumerate(tokens_ex):
    ax.add_patch(plt.Circle((i, 0.5), 0.32, color='#4C72B0', zorder=3))
    ax.text(i, 0.5, tok, ha='center', va='center', color='white', fontsize=8.5, fontweight='bold')

# "it" attends strongly to "trophy", weakly to others
strengths = [0.03, 0.80, 0.05, 0.05, 0.07]
for j in range(n-1):
    lw = max(0.3, strengths[j] * 5)
    color = 'crimson' if j == 1 else '#aaaaaa'
    ax.annotate("", xy=(j+0.33, 0.72), xytext=(5-0.33, 0.72),
                arrowprops=dict(arrowstyle='<-', color=color, lw=lw,
                                connectionstyle=f'arc3,rad={-0.25 - j*0.04}'))
ax.text(3, -0.22, '"it" attends directly to "trophy" in one step', ha='center', color='crimson', fontsize=9)
ax.set_xlim(-0.7, n-0.3); ax.set_ylim(-0.4, 1.2); ax.axis('off')

plt.tight_layout()
plt.show()

<center>

---
---

# 🟢 Part 2: The Attention Mechanism

---
---
</center>

## 2.1  Intuition: a soft dictionary lookup

Imagine a Python dictionary:
```python
{"cat": "furry animal", "dog": "loyal animal", "fish": "aquatic animal"}
```
A *hard* lookup for `"cat"` returns exactly one value.

**Attention is a *soft* lookup** — given a query $Q$, it returns a *weighted average* of all values, where the weights depend on how well each key $K$ matches the query:

$$\boxed{\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V}$$

| Symbol | Role | Intuition |
|--------|------|-----------|
| $Q$ | Query | "What am I looking for?" |
| $K$ | Key | "What does each position offer?" |
| $V$ | Value | "What information do I actually retrieve?" |
| $d_k$ | Key dimension | Used to prevent dot products from growing too large |

Given input embeddings $X \in \mathbb{R}^{n \times d}$, three learned linear projections produce:

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

where $W^Q, W^K, W^V \in \mathbb{R}^{d \times d_k}$ are trained by gradient descent.

## 2.2  Step-by-step in NumPy

We use a toy sentence: `["The", "cat", "sat", "mat"]` with random embeddings.

In [ ]:
# ── Toy example: 4 tokens, embedding dimension = 8 ───────────────────────────
tokens = ["The", "cat", "sat", "mat"]
T = len(tokens)    # sequence length
d_model = 8        # embedding dimension
d_k = 4            # key/query dimension (often d_model // num_heads)

# Pretend these are learned embedding vectors for each token
X = np.random.randn(T, d_model)

# Learned projection matrices — normally trained by gradient descent, here random
W_Q = np.random.randn(d_model, d_k)
W_K = np.random.randn(d_model, d_k)
W_V = np.random.randn(d_model, d_k)

# Project inputs → Q, K, V
Q = X @ W_Q   # (T, d_k)
K = X @ W_K
V = X @ W_V

print(f"Input X:  {X.shape}   (sequence_length, d_model)")
print(f"Q shape:  {Q.shape}   (sequence_length, d_k)")
print(f"K shape:  {K.shape}")
print(f"V shape:  {V.shape}")

In [ ]:
# ── Step 1: raw attention scores  QKᵀ / √d_k ─────────────────────────────────
scores = Q @ K.T / np.sqrt(d_k)   # (T, T)
print("Raw scores (before softmax):")
print(np.round(scores, 2))
print(f"\nShape: {scores.shape}  — one scalar per (query, key) pair")

In [ ]:
# ── Step 2: softmax → attention weights ──────────────────────────────────────
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

weights = softmax(scores)   # (T, T)
print("Attention weights (each row is a probability distribution):")
print(np.round(weights, 3))
print("\nRow sums:", np.round(weights.sum(axis=1), 6))

In [ ]:
# ── Step 3: weighted sum of values ───────────────────────────────────────────
output = weights @ V   # (T, d_k)
print("Attention output shape:", output.shape)
print("Output for 'The'  :", np.round(output[0], 3))
print("Output for 'cat'  :", np.round(output[1], 3))

In [ ]:
# ── Visualise the attention weight matrix ────────────────────────────────────
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(T)); ax.set_xticklabels(tokens, fontsize=11)
ax.set_yticks(range(T)); ax.set_yticklabels(tokens, fontsize=11)
ax.set_xlabel("Keys (attended TO)", fontsize=11)
ax.set_ylabel("Queries (attending FROM)", fontsize=11)
ax.set_title("Attention weight matrix\n[\"The cat sat mat\" — random weights]", fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.046)
for i in range(T):
    for j in range(T):
        ax.text(j, i, f"{weights[i,j]:.2f}", ha='center', va='center',
                fontsize=9, color='white' if weights[i,j] > 0.5 else 'black')
plt.tight_layout()
plt.show()

> **Check ✏️**
> - Each *row* is the attention pattern for one query token. Which token attends most strongly to itself?
> - What does it mean when a token has high attention weight on a *different* token?
> - Why do we divide by $\sqrt{d_k}$? Try removing the division and print the raw scores again — what happens to the softmax output? (Hint: think about gradient saturation.)

## 2.3  Causal masking — preventing the future from leaking

In **encoder** models (e.g. BERT), every token can attend to every other — fine for understanding tasks.

In **decoder** models (e.g. GPT), we generate text left to right. Token at position $t$ must not look at $t+1, t+2, \ldots$ — that would be cheating.

We enforce this with a **causal mask**: set future positions to $-\infty$ before the softmax, so $e^{-\infty} = 0$:

$$\text{score}_{ij} \leftarrow -\infty \quad \text{if } j > i$$

In [ ]:
# ── Causal mask ───────────────────────────────────────────────────────────────
mask = np.tril(np.ones((T, T)))           # lower triangular: 1 = allowed
mask_val = np.where(mask == 0, -1e9, 0)   # 0 = keep, -1e9 ≈ -∞

masked_scores = scores + mask_val
causal_weights = softmax(masked_scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, w, title, cmap in zip(
    axes,
    [weights, causal_weights],
    ['Full attention (encoder / BERT-style)', 'Causal mask (decoder / GPT-style)'],
    ['Blues', 'Oranges']
):
    im = ax.imshow(w, cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(T)); ax.set_xticklabels(tokens, fontsize=10)
    ax.set_yticks(range(T)); ax.set_yticklabels(tokens, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel("Key"); ax.set_ylabel("Query")
    for i in range(T):
        for j in range(T):
            ax.text(j, i, f"{w[i,j]:.2f}", ha='center', va='center',
                    fontsize=9, color='white' if w[i,j] > 0.5 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print("Encoder: bidirectional → good for classification, NER, question answering")
print("Decoder: causal        → good for text generation, code, reasoning")

### The running example with a semantically meaningful attention matrix

With random projections the weights are arbitrary. Here is what a *trained* model might produce for the trophy sentence — where *"it"* correctly attends to *"trophy"*:

In [ ]:
# Illustrative (hand-crafted) attention weights for a trained model
tokens_trophy = ["The", "trophy", "didn't", "fit", "because", "it"]
n2 = len(tokens_trophy)

attn_trained = np.array([
    [0.50, 0.15, 0.10, 0.10, 0.10, 0.05],  # "The"
    [0.05, 0.55, 0.10, 0.10, 0.10, 0.10],  # "trophy"
    [0.10, 0.15, 0.50, 0.10, 0.10, 0.05],  # "didn't"
    [0.05, 0.10, 0.05, 0.55, 0.15, 0.10],  # "fit"
    [0.05, 0.10, 0.10, 0.05, 0.60, 0.10],  # "because"
    [0.05, 0.70, 0.05, 0.05, 0.10, 0.05],  # "it" → strongly attends to "trophy"
])

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(attn_trained, annot=True, fmt=".2f",
            xticklabels=tokens_trophy, yticklabels=tokens_trophy,
            cmap='Blues', vmin=0, vmax=1, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title('Illustrative attention — trained model\n"The trophy didn\'t fit because it was too big"', pad=10)
ax.set_xlabel("Key (source of information)")
ax.set_ylabel("Query (consumer)")
ax.add_patch(plt.Rectangle((1, 5), 1, 1, fill=False, edgecolor='crimson', lw=3))
ax.text(1.5, 4.72, '"it" → "trophy"', color='crimson', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

<center>

---
---

# 🟢 Part 3: Multi-Head Attention

---
---
</center>

## 3.1  Why multiple heads?

A single attention head learns one way of relating tokens. Language requires many simultaneous relationship types:

- **Syntactic:** subject → verb agreement
- **Semantic / coreference:** pronoun → antecedent (*"it"* → *"trophy"*)
- **Positional:** nearby tokens tend to be related
- **Structural:** matching delimiters, clause boundaries

**Multi-head attention** runs $h$ independent attention operations in parallel:

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\, W^O$$

where $\quad \text{head}_i = \text{Attention}(Q W_i^Q,\, K W_i^K,\, V W_i^V)$

To keep computation constant, each head uses $d_k = d_{\text{model}} / h$ (e.g. 12 heads × $d_k = 64$ = $d_{\text{model}} = 768$ in BERT-base).

In [ ]:
def attention_fn(Q, K, V, mask=None):
    """Scaled dot-product attention."""
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)
    if mask is not None:
        scores += mask
    w = softmax(scores)
    return w @ V, w

def multihead_attention(X, num_heads, d_model, mask=None):
    """Minimal multi-head attention (numpy, no training)."""
    assert d_model % num_heads == 0
    d_k = d_model // num_heads
    T = X.shape[0]
    all_outputs, all_weights = [], []
    for h in range(num_heads):
        Wq = np.random.randn(d_model, d_k) * 0.1
        Wk = np.random.randn(d_model, d_k) * 0.1
        Wv = np.random.randn(d_model, d_k) * 0.1
        out_h, w_h = attention_fn(X @ Wq, X @ Wk, X @ Wv, mask)
        all_outputs.append(out_h)
        all_weights.append(w_h)
    concat = np.concatenate(all_outputs, axis=-1)   # (T, d_model)
    W_O = np.random.randn(d_model, d_model) * 0.1
    return concat @ W_O, all_weights

np.random.seed(7)
num_heads = 2
X_mha = np.random.randn(T, d_model)
output_mha, head_weights = multihead_attention(X_mha, num_heads, d_model)
print(f"Multi-head attention output shape: {output_mha.shape}")
print(f"Number of heads: {num_heads}, d_k per head: {d_model // num_heads}")

In [ ]:
# ── Visualise each head's attention pattern ───────────────────────────────────
fig, axes = plt.subplots(1, num_heads, figsize=(5 * num_heads, 4.5))
for h, (ax, w) in enumerate(zip(axes, head_weights)):
    im = ax.imshow(w, cmap='Purples', vmin=0, vmax=1)
    ax.set_xticks(range(T)); ax.set_xticklabels(tokens, fontsize=10)
    ax.set_yticks(range(T)); ax.set_yticklabels(tokens, fontsize=10)
    ax.set_title(f"Head {h+1}", fontsize=12)
    for i in range(T):
        for j in range(T):
            ax.text(j, i, f"{w[i,j]:.2f}", ha='center', va='center',
                    fontsize=9, color='white' if w[i,j] > 0.5 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("Each head learns a different attention pattern", fontsize=12)
plt.tight_layout()
plt.show()

### What different heads actually learn in real models

Here are three stylised patterns observed in trained models (Clark et al., 2019; Voita et al., 2019):

In [ ]:
tokens_s = ["The", "trophy", "didn't", "fit", "it"]
n3 = len(tokens_s)

head1 = np.array([  # positional — adjacent tokens
    [0.60, 0.30, 0.05, 0.03, 0.02],
    [0.30, 0.40, 0.20, 0.05, 0.05],
    [0.05, 0.20, 0.50, 0.20, 0.05],
    [0.03, 0.05, 0.20, 0.55, 0.17],
    [0.02, 0.05, 0.05, 0.17, 0.71],
])
head2 = np.array([  # coreference — "it" → "trophy"
    [0.70, 0.10, 0.10, 0.05, 0.05],
    [0.10, 0.60, 0.10, 0.10, 0.10],
    [0.10, 0.10, 0.60, 0.10, 0.10],
    [0.05, 0.10, 0.10, 0.65, 0.10],
    [0.03, 0.80, 0.05, 0.07, 0.05],
])
head3 = np.array([  # syntactic — subject-verb linkage
    [0.50, 0.05, 0.35, 0.05, 0.05],
    [0.05, 0.50, 0.05, 0.35, 0.05],
    [0.35, 0.05, 0.50, 0.05, 0.05],
    [0.05, 0.35, 0.05, 0.50, 0.05],
    [0.05, 0.05, 0.05, 0.05, 0.80],
])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, H, title, cmap in zip(
    axes,
    [head1, head2, head3],
    ["Head 1: Positional\n(adjacent tokens)",
     "Head 2: Coreference\n(\"it\" → \"trophy\")",
     "Head 3: Syntactic\n(subject-verb)"],
    ['Blues', 'Reds', 'Greens']
):
    sns.heatmap(H, annot=True, fmt=".2f",
                xticklabels=tokens_s, yticklabels=tokens_s,
                cmap=cmap, vmin=0, vmax=1, ax=ax,
                linewidths=0.5, linecolor='white')
    ax.set_title(title, fontweight='bold', pad=8)

plt.suptitle("Stylised specialisation of attention heads in a trained model",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

> **Check ✏️**
> - Do the two randomly-initialised heads above produce the same pattern? Why or why not?
> - In GPT-4 there are an estimated 96 attention heads. What might each head specialise in?
> - Some heads in trained models are found to be **prunable** (removing them causes no performance drop). What does this suggest about redundancy in multi-head attention?

<center>

---
---

# 🟢 Part 4: The Transformer Block

---
---
</center>



####  The Core Problem

Before Transformers, models like RNNs processed words one by one, which was slow and made it hard to track long-range dependencies (e.g., linking a pronoun to a noun 20 words earlier). Transformers solved this with the **attention mechanism**, allowing them to look at all words in a sequence simultaneously.

####  Building Blocks

A Transformer block is a "sandwich" of two main layers, each wrapped in a **residual connection** and **layer normalization** for training stability.

**Step 1: Multi-Head Self-Attention (The "Look Around" Layer)**

This layer enriches each token's representation by analyzing its relationship to *all* tokens in the sequence. It answers the question: "To understand this word, which other words in the sentence are most relevant, and why?"

*   **Key Insight**: It mixes information *across* the sequence (mixing tokens).

**Step 2: Feed-Forward Network (The "Think" Layer)**

After attention, each token's representation is processed independently by a small, 2-layer neural network. This layer acts as a **"knowledge store"** and applies complex, non-linear transformations to the data.


**Step 3: The "Glue" (Residual Connections & Layer Norm)**

*   **Residual Connections**: These are "skip connections" that add the input of a sub-layer to its output. They help in training very deep networks by providing a direct path for the gradient flow, mitigating the vanishing gradient problem (similar to U-net).
*   **Layer Normalization (Pre-LN)**: This stabilizes training by normalizing the inputs to each sub-layer to have a mean of 0 and a standard deviation of 1, controlling the scale of the values. The "Pre-LN" variant is standard in modern models (e.g., GPT, LLaMA).

<image src="https://raw.githubusercontent.com/marcinwolter/MachineLearning-KISD-2026/refs/heads/main/images/deepseek_mermaid_20260420_5faf18.png" width=400px>



####  The Complete Flow

Here'
s the data flow through one Transformer block (the "Pre-LN" variant used in most modern LLMs):

1.  **Input**: The input to the block is the output from the previous block (or the initial token + positional embeddings).
2.  **Attention Path**: The input `x` is first normalized. This normalized version goes through the Multi-Head Attention layer. The output is added to the original `x` via a residual connection.
3.  **FFN Path**: The output from the attention path (`x1`) is normalized again, then passed through the FFN. The FFN's output is added back to `x1` via a second residual connection.
4.  **Output**: The final output (`x2`) is then sent to the next Transformer block (or to the final output layer).

This can be expressed with the following equations:
*   `x1 = x + MultiHeadAttention(LayerNorm(x))`
*   `x2 = x1 + FFN(LayerNorm(x1))`

####  Intuition: The "Residual Stream"

You can think of the input as a **"residual stream" of information**. Each sub-layer reads from this stream, computes an **"update"** (or a "correction"), and adds it back. The original information is never overwritten but is continuously refined.

####  Summary

*   **Attention**: Mixes information *across* tokens to gather context.
*   **FFN**: Mixes information *within* a token to "think" about the new context.
*   **Residual Connections**: Prevent information loss and enable deep networks.
*   **Layer Normalization**: Keeps training stable and efficient.



<center>

---
---

# 🟢 Part 5: From Transformer to Large Language Models

---
---
</center>

## 5.1  Language modelling

An LLM is trained on a deceptively simple task:

> **Given the previous tokens, predict the next token.**

$$P(w_t \mid w_1, w_2, \ldots, w_{t-1})$$

This is called **autoregressive language modelling**. Training objective (cross-entropy loss):

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log P(w_t \mid w_1, \ldots, w_{t-1})$$

What makes LLMs powerful is not the objective but the **scale**: trillions of tokens, billions of parameters.



## 5.2  GPT architecture

GPT (Generative Pre-trained Transformer) is a **decoder-only** Transformer:

```
Input tokens  →  Token embedding + Positional encoding
                      │
              ┌───────▼──────────┐
              │ Transformer      │
              │ block × N        │  ← causal mask applied here
              └───────┬──────────┘
                      │
              Linear projection → vocabulary logits
                      │
                   Softmax → P(next token)
```

| Model | Layers | Heads | $d_{\text{model}}$ | Parameters |
|-------|--------|-------|---------|------------|
| GPT-1 | 12 | 12 | 768 | 117M |
| GPT-2 | 48 | 25 | 1600 | 1.5B |
| GPT-3 | 96 | 96 | 12288 | 175B |
| LLaMA 3 70B | 80 | 64 | 8192 | 70B |
| GPT-4 (est.) | ~120 | ~96 | ~25600 | ~1.8T |

## 5.3  Post-training alignment

A pre-trained model is a powerful next-token predictor but not necessarily a useful assistant. Post-training stages shape it toward helpfulness and safety:

1. **Supervised Fine-Tuning (SFT):** Fine-tune on curated instruction-following examples
2. **RLHF / DPO:** Use human preference data to further align outputs

## 5.4  Scaling laws

Kaplan et al. (2020) and Hoffmann et al. (2022, *Chinchilla*) established that loss scales as a power law in model size $N$ and data $D$:

$$\mathcal{L} \propto N^{-\alpha}, \quad \mathcal{L} \propto D^{-\beta}$$

The Chinchilla finding: for a given compute budget $C$, optimal training uses roughly **20 tokens per parameter** — most earlier models were significantly undertrained on data.

<center>

---
---

# 🟢 Part 6: Hands-on — Tokenisation, Embeddings & a Tiny GPT

---
---
</center>

## 6.1  What is a token?

LLMs do not process characters or words — they process **tokens**: subword units produced by Byte-Pair Encoding (BPE).

```
"unhappiness"  →  ["un", "happ", "iness"]
"ChatGPT"      →  ["Chat", "G", "PT"]
"2026"         →  ["2026"]          (common numbers are often one token)
```

Tokens allow the vocabulary to stay small (~50,000) while handling any word, including rare ones and new proper nouns.

In [ ]:
# Install the tiktoken library (OpenAI's fast tokeniser, used by GPT-2/3/4)
!pip install tiktoken -q

import tiktoken
enc = tiktoken.get_encoding("gpt2")   # GPT-2 vocabulary, 50 257 tokens

examples = [
    "Hello world!",
    "The attention mechanism is the key innovation.",
    "unhappiness",
    "ChatGPT",
    "Kraków 2026",
    "x_train.reshape(60000, 784)",
]

print(f"{'Text':<45} {'Tokens':>7}  Token pieces")
print("─" * 80)
for text in examples:
    ids = enc.encode(text)
    decoded = [enc.decode([i]) for i in ids]
    print(f"{text:<45} {len(ids):>7}  {decoded}")

In [ ]:
# ── Visualise token boundaries in a longer sentence ──────────────────────────
def getTokens(sentence):
  ids = enc.encode(sentence)
  pieces = [enc.decode([i]) for i in ids]

  print(f"Original : {sentence}")
  print(f"Num tokens: {len(ids)}")
  print("\nTokens:")
  colors_term = ['\033[44m', '\033[42m', '\033[43m', '\033[41m', '\033[45m', '\033[46m']
  reset = '\033[0m'
  highlighted = "".join(colors_term[i % len(colors_term)] + p + reset for i, p in enumerate(pieces))
  print(highlighted)

In [ ]:
sentence = "Large language models are trained on next-token prediction at massive scale."
getTokens(sentence)

sentence = "The perpetually disenchanted, overindustrialized, and hypercompetitive\
  doctoral researchers — despite their chronic sleep deprivation and multifarious\
  existential anxieties — remain the quintessentially idealistic, albeit neurotically\
  overachieving, candidates for the internationally preeminent, futuristically\
  conjectured Nobel laureate distinction."

getTokens(sentence)


## 6.2 Word Embeddings: From Symbols to Meaningful Vectors

### The problem: Computers don't understand words

A neural network cannot directly process the word `"cat"`. It only works with numbers. The simplest solution is **one‑hot encoding**: assign a unique integer to each word, then represent that integer as a binary vector with a single `1`.

Example:  
`"cat"` → `[0, 0, 1, 0, ..., 0]`  
`"dog"` → `[0, 1, 0, 0, ..., 0]`

**Problem**: These vectors are orthogonal (all pairs have dot product zero). They imply `"cat"` and `"dog"` are as unrelated as `"cat"` and `"airplane"`. No notion of *similarity* or *meaning*.

### The solution: Dense, learned embeddings

Instead of a sparse binary vector, we map each word to a **dense, low‑dimensional vector** of real numbers (e.g., 300 dimensions). These vectors are **learned from data** so that:

- **Semantically similar words** have similar vectors (they are close in vector space).
- **Relationships** can be captured by vector arithmetic (e.g., `"king" - "man" + "woman" ≈ "queen"`).

### Intuition with an analogy

Think of each word as a point in a high‑dimensional "meaning space".  
- Dimensions represent latent features like *gender*, *size*, *emotion*, *tense*, *formality*, etc.  
- The network discovers these features automatically—no one labels them.

For example, a 3‑D embedding of animal words might place:

| Word   | dimension₁ (furry?) | dimension₂ (domestic?) | dimension₃ (size) |
|--------|---------------------|------------------------|-------------------|
| cat    | 0.9                 | 0.8                    | -0.2              |
| dog    | 0.8                 | 0.9                    | -0.1              |
| elephant | 0.7               | -0.5                   | 0.9               |
| airplane | -0.9              | -0.8                   | 0.4               |

Now `"cat"` and `"dog"` have similar vectors → they are close in space. `"cat"` and `"airplane"` are far apart.

### How are they learned?

The embeddings are just **trainable parameters**—a matrix of size `(vocab_size, embedding_dim)`. Initially random. During training, the model adjusts them to minimise a loss function (e.g., predict the next word). Words that appear in similar contexts get pulled together.

**Example training signal**:  
In the sentence *"The ___ sat on the mat"*, if the model sees both `"cat"` and `"dog"` in that blank across many examples, it learns to give them similar embeddings.

### Why does this matter?

- **Semantic similarity** → the model can generalise: if it knows `"cat likes milk"`, it can guess `"dog likes ?"` (bones, milk? – but at least something positive).
- **Transfer learning**: Pre‑trained embeddings (Word2Vec, GloVe, or embeddings inside LLMs) encode rich world knowledge without explicit rules.
- **Efficiency**: Dense vectors (e.g., 768 dimensions) are far more compact than one‑hot vectors (e.g., 50,000 dimensions).

### Key properties to remember

| One‑hot encoding | Dense word embedding |
|------------------|----------------------|
| High‑dimensional (vocab size) | Low‑dimensional (e.g., 300) |
| Sparse (mostly zeros) | Dense (all entries non‑zero) |
| No notion of similarity | Similar words → similar vectors |
| Fixed, not learned | Learned from data |
| Orthogonal vectors | Vectors with meaningful angles |


In [ ]:
# ── Explore embeddings with GloVe (pre-trained static embeddings) ─────────────
!pip install gensim -q

from gensim.downloader import load as gensim_load
import warnings; warnings.filterwarnings('ignore')

print("Downloading GloVe-50 embeddings (~70 MB)...")
glove = gensim_load("glove-wiki-gigaword-50")
print(f"Vocabulary size: {len(glove):,}  |  Embedding dimension: {glove.vector_size}")

The famous example: $\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$

In [ ]:
# ── Word arithmetic ───────────────────────────────────────────────────────────
print("king - man + woman =")
for word, score in glove.most_similar(positive=['king', 'woman'], negative=['man'], topn=5):
    print(f"  {word:<15} {score:.3f}")

print()
print("Paris - France + Poland =")
for word, score in glove.most_similar(positive=['paris', 'poland'], negative=['france'], topn=5):
    print(f"  {word:<15} {score:.3f}")

print()
print("Most similar to 'neural':")
for word, score in glove.most_similar('neural', topn=6):
    print(f"  {word:<15} {score:.3f}")

In [ ]:
# ── Visualise embeddings with PCA ─────────────────────────────────────────────
from sklearn.decomposition import PCA
import numpy as np

words_pca = ['king', 'queen', 'man', 'woman', 'boy', 'girl',
             'dog', 'cat', 'fish', 'bird',
             'paris', 'london', 'berlin', 'warsaw', 'rome',
             'france', 'england', 'germany', 'poland', 'italy']
vectors = np.array([glove[w] for w in words_pca])
coords = PCA(n_components=2).fit_transform(vectors)

groups = {'royals/gender': words_pca[:6], 'animals': words_pca[6:10],
          'capitals': words_pca[10:15], 'countries': words_pca[15:]}
group_colors = {'royals/gender': '#378ADD', 'animals': '#1D9E75',
                'capitals': '#D85A30', 'countries': '#9C59B6'}

fig, ax = plt.subplots(figsize=(10, 7))
for group, members in groups.items():
    idxs = [words_pca.index(w) for w in members]
    ax.scatter(coords[idxs, 0], coords[idxs, 1],
               c=group_colors[group], label=group, s=80, zorder=3)
for i, word in enumerate(words_pca):
    ax.annotate(word, coords[i], fontsize=9, xytext=(4, 4), textcoords='offset points')
ax.set_title("GloVe word embeddings projected to 2D (PCA)", fontsize=12)
ax.legend(fontsize=10); ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

> **Check ✏️**
> - Do capital cities cluster together? Do countries? What does this tell you about what GloVe learned?
> - Find a word pair where the analogy *word_A - word_B + word_C* gives a surprising or wrong result.




## 6.3  Build a tiny GPT in Keras

We will train a **character-level** language model on *Pride and Prejudice*. This is not a real LLM — it has ~80K parameters and trains in minutes — but it demonstrates every architectural component of GPT.

**Architecture:** exactly the same as GPT, just with characters as tokens and a tiny $d_{\text{model}}$.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import urllib.request

# Download Pride and Prejudice (Project Gutenberg)
url = "https://www.gutenberg.org/files/1342/1342-0.txt"
urllib.request.urlretrieve(url, "pride.txt")

with open("pride.txt", encoding="utf-8") as f:
    text_full = f.read()

# Ensure text_full is not empty; if it is, use a placeholder
if not text_full.strip():
    print("Warning: Downloaded text is empty or only whitespace. Using placeholder text.")
    text = "The quick brown fox jumps over the lazy dog. This is a sample text for training."
else:
    # Original logic for slicing the text
    start_phrase = "Their sister's wedding-day arrived"
    end_phrase   = "Derbyshire, had been the means of uniting them."

    start_idx = text_full.find(start_phrase)
    end_idx   = text_full.rfind(end_phrase)

    # Use the full text if markers are not found or range is invalid
    if start_idx == -1 or end_idx == -1 or start_idx >= end_idx:
        print(f"Warning: Text markers '{start_phrase}' or '{end_phrase}' not found or invalid range. Using full text for vocabulary.")
        text = text_full
    else:
        # +len(end_phrase) to ensure the end phrase is fully included
        text = text_full[start_idx : end_idx + len(end_phrase)]

print(f"Text length: {len(text):,} characters")
print(f"First 200 chars:\n{text[:200]}")

In [ ]:
# ── Character-level vocabulary ────────────────────────────────────────────────
chars = sorted(set(text))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char  = {i: c for c, i in char2idx.items()}

data = np.array([char2idx[c] for c in text], dtype=np.int32)

print(f"Vocabulary size: {vocab_size} unique characters")
print(f"Encoded length:  {len(data):,} tokens")
print(f"First 30 tokens: {data[:30]}")

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
SEQ_LEN    = 64     # context window length
BATCH_SIZE = 64
EMBED_DIM  = 64     # d_model
NUM_HEADS  = 4
FF_DIM     = 128    # feed-forward hidden size (2× d_model)
NUM_LAYERS = 2      # number of Transformer blocks
DROPOUT    = 0.1

def make_dataset(data, seq_len, batch_size):
    """Create (input, target) pairs where target is input shifted by 1."""
    ds = tf.data.Dataset.from_tensor_slices(data)
    ds = ds.window(seq_len + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda w: w.batch(seq_len + 1))
    ds = ds.map(lambda x: (x[:-1], x[1:]))
    ds = ds.shuffle(10000).batch(batch_size, drop_remainder=True)
    return ds.prefetch(tf.data.AUTOTUNE)

dataset = make_dataset(data, SEQ_LEN, BATCH_SIZE)
print(f"Dataset ready — input/target shape per batch: {BATCH_SIZE}×{SEQ_LEN}")

In [ ]:
# ── Transformer block ─────────────────────────────────────────────────────────
def transformer_block(x, embed_dim, num_heads, ff_dim, dropout):
    # Multi-head self-attention with causal mask (use_causal_mask=True → GPT-style)
    attn = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=dropout
    )(x, x, use_causal_mask=True)
    x = layers.LayerNormalization()(x + attn)   # residual + LN

    # Feed-forward network
    ff = layers.Dense(ff_dim, activation='relu')(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(embed_dim)(ff)
    x = layers.LayerNormalization()(x + ff)     # residual + LN
    return x

# ── Build tiny GPT ────────────────────────────────────────────────────────────
inputs  = keras.Input(shape=(SEQ_LEN,), dtype='int32')
tok_emb = layers.Embedding(vocab_size, EMBED_DIM)(inputs)          # token embedding
pos_emb = layers.Embedding(SEQ_LEN,   EMBED_DIM)(tf.range(SEQ_LEN))  # learned positional embedding
x = layers.Dropout(DROPOUT)(tok_emb + pos_emb)

for _ in range(NUM_LAYERS):
    x = transformer_block(x, EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT)

outputs = layers.Dense(vocab_size)(x)   # logits over character vocabulary

model = keras.Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
model.summary()

Warning: the model should be trained for much more epochs. Just 3 epochs to save the time.

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
from keras.callbacks import EarlyStopping

history = model.fit(
    dataset,
    epochs=3,
    callbacks=[EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)],
    verbose=1
)

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], 'b-o', ms=4)
ax1.set(xlabel='Epoch', ylabel='Loss', title='Training loss'); ax1.grid(alpha=0.3)
ax2.plot(history.history['accuracy'], 'r-o', ms=4)
ax2.set(xlabel='Epoch', ylabel='Accuracy', title='Training accuracy'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Text generation: temperature sampling ────────────────────────────────────
def generate(model, prompt, num_chars=300, temperature=1.0):
    """
    Generate text autoregressively one character at a time.
    temperature < 1 → more conservative/repetitive
    temperature > 1 → more creative/random
    """
    result = list(prompt)
    for _ in range(num_chars):
        context = result[-SEQ_LEN:]
        x = np.array([[char2idx.get(c, 0) for c in context]])
        x = np.pad(x, [(0, 0), (SEQ_LEN - x.shape[1], 0)])  # left-pad
        logits = model.predict(x, verbose=0)[0, -1] / temperature
        probs  = tf.nn.softmax(logits).numpy()
        next_char = np.random.choice(len(chars), p=probs)
        result.append(idx2char[next_char])
    return ''.join(result)

prompt = "It is a truth universally acknowledged"

for temp in [0.5, 1.0, 1.5]:
    print(f"{'─'*60}")
    print(f"Temperature = {temp}:")
    print(generate(model, prompt, temperature=temp))
    print()

In [ ]:
prompt = "Our Institute of Physics and the PhD students are world famous of "

for temp in [0.5, 1.0, 1.5]:
    print(f"{'─'*60}")
    print(f"Temperature = {temp}:")
    print(generate(model, prompt, temperature=temp))
    print()

> **Check ✏️ — Temperature**
> - At temperature 0.5, does the text look more or less like English than at 1.5?
> - What happens as temperature → 0 (greedy decoding)? What happens as temperature → ∞?
> - Real LLMs use **top-k** and **top-p (nucleus)** sampling in addition to temperature. Why? What failure mode does each address?


<center>

---
---

# 🟢 Summary

---
---
</center>

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                     Lecture — Key Concepts                       ║
╠══════════════════════════════════════════════════════════════════╣
║  Attention          softmax(QKᵀ/√dₖ) V                         ║
║  Q, K, V            learned projections of input embeddings     ║
║  Causal mask        future tokens blocked (GPT/decoder)         ║
║  Multi-head         parallel attention with different focuses    ║
║  Positional enc.    sin/cos or learned (RoPE in modern LLMs)    ║
║  Transformer block  LayerNorm + MHA + residual + FFN + residual ║
║  GPT                decoder-only, next-token prediction         ║
║  Tokenisation       subword BPE, ~50K vocabulary                ║
║  Static embeddings  GloVe — context-independent                 ║
║  Contextual emb.    Transformer — same word, different vector   ║
║  Temperature        controls randomness at generation time      ║
║  Scaling laws       loss ~ N^-α ~ D^-β (Chinchilla)            ║
╚══════════════════════════════════════════════════════════════════╝
""")



---



---



---




#<font color='green'>**Regression with neural networks**



---



---



---




Up to now we were dealing with classification. Let's talk about regression.

<img src='https://upload.wikimedia.org/wikipedia/commons/b/be/Normdist_regression.png' width=250px>

**Regression analysis is a set of statistical processes for estimating the relationships between a dependent variable or variables and one or more independent variables.**

We can perform regression using deep neural network:



In [ ]:
# Simple 1-D keras regression

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np


# Keras specific
import keras
from keras.models import Sequential
from keras.layers import Dense

import matplotlib.pyplot as plt

# Fitted function y=f(x)
X = np.random.uniform(-4,4,200)
y = np.sin(X)*np.exp(0.01*X)+np.cos(X)
err = np.random.normal(0,0.1,200)
y = y + err

# Function x=f(y)
#y = np.random.uniform(-4,4,200)
#X = np.sin(y)*np.exp(0.01*y)+np.cos(y)
#err = np.random.normal(0,0.1,200)
#y = y + err

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    random_state=1)

# Define model
model = Sequential()
model.add(Dense(256, input_dim=1, activation= "relu"))
model.add(Dense(512, activation= "relu"))
# VERY IMPORTANT! The last layer should have linear activation function
# to cover the entire output range
model.add(Dense(1, activation="linear"))
model.summary() #Print model Summary

model.compile(loss= "mse" , optimizer="adam", metrics=["mean_squared_error"])
history = model.fit(X_train, y_train, validation_data=(X_test,y_test), epochs=60)

pred_train= model.predict(X_train)
print("Train dataset error: ",np.sqrt(mean_squared_error(y_train,pred_train)))

pred= model.predict(X_test)
print("Test dataset error: ",np.sqrt(mean_squared_error(y_test,pred)))

scores = model.evaluate(X_test, y_test)
print(scores)


### Plot the loss

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['loss','val_loss'])
plt.show()

### Plot the fitted function

In [ ]:
from sklearn.utils.fixes import linspace

fig = plt.figure()

plt.scatter(X_test,y_test, marker='o')

xx = linspace(min(X_test), max(X_test), num = 200)
plt.plot(xx, model.predict(xx),color='red')

plt.show()

#**Problems:**
 -  No information about errors of the fitted function,
 -  No regression to non-functional dependence:
<img src='https://raw.githubusercontent.com/marcinwolter/NORCC-SUMMER_2022/main/images/reversed_fit.png' width=350px>

Here we have modified the generated data to:

```python
# Function x=f(y)
y = np.random.uniform(-4,4,NSAMPLE)
X = np.sin(y)*np.exp(0.01*y)+np.cos(y)
err = np.random.normal(0,0.3,NSAMPLE)
y = y + err
```

# <font color='green'> Machine Learning vs. Bayesian Learning </font>

| Machine Learning | Bayesian Learning |
|:----------------:|:-----------------:|
| <img src='https://raw.githubusercontent.com/marcinwolter/NORCC-SUMMER_2022/main/images/Screenshot%20from%202022-06-18%2021-01-30.png' width=300px> | <img src='https://raw.githubusercontent.com/marcinwolter/NORCC-SUMMER_2022/main/images/Screenshot%20from%202022-06-18%2021-03-33.png' width=300px> |
| **We choose one function** (or a single set of parameters) that best fits the training data. | **Each possible function (or parameter set) is assigned a probability** reflecting how likely it is given the data and prior beliefs. |
| *Point estimate*: The goal is to find a single optimal model (e.g., via loss minimisation). | *Full posterior distribution*: The goal is to compute or approximate the distribution over all models. |
| **Example**: Ordinary least squares regression gives one line of best fit. | **Example**: Bayesian linear regression gives a distribution over lines, with uncertainty intervals. |

<center>

<img src='https://raw.githubusercontent.com/marcinwolter/NORCC-SUMMER_2022/main/images/Thomas_Bayes.gif' width=250px>

</center>

**Rev. Thomas Bayes** (c. 1701 – 7 April 1761 was an English statistician, philosopher and Presbyterian minister who is known for formulating a theorem that bears his name: **Bayes' theorem**.

# **Mixture Density Network**

A network returning the parametrized probability distribution.
The output is parametrized by few Gaussian distributions.

<img src='https://raw.githubusercontent.com/marcinwolter/NORCC-SUMMER_2022/main/images/MDN.png' width=350px>

<img src='https://raw.githubusercontent.com/marcinwolter/NORCC-SUMMER_2022/main/images/Screenshot%20from%202022-06-18%2021-57-58.png' width=650px>

*Bishop, Christopher M. Mixture density networks. Technical Report NCRG/4288,Aston University, Birmingham, UK, 1994*<br>
*https://publications.aston.ac.uk/id/eprint/373/1/NCRG_94_004.pdf*

Nice presentation about mixture density networks:<br>
*https://www.dbs.ifi.lmu.de/Lehre/DLAI/WS18-19/script/06_uncertain.pdf*



# **How to build MDN?**

We use the MDN implementation from:<br> http://github.com/cpmpercussion/keras-mdn-layer.

So, the modifications needed:

1. Install keras-mdn-layer
```python
!pip install keras-mdn-layer
import mdn
```
2. Add mdn layer in network declaration and a special loss function:
```python
N_MIXES = 5
#
model = keras.Sequential()
model.add(keras.layers.Dense(N_HIDDEN, batch_input_shape=(None, 1), activation='relu'))
model.add(keras.layers.Dense(N_HIDDEN, activation='relu'))
model.add(mdn.MDN(1, N_MIXES))
#
model.compile(loss=mdn.get_mixture_loss_func(1,N_MIXES), optimizer=keras.optimizers.Adam()) #, metrics=[mdn.get_mixture_mse_accuracy(1,N_MIXES)])
model.summary()
```


# Mixture Density Network

Reproducing the classic Bishop MDN network tasks in Keras. The idea in this task is to predict a the value of an inverse sine function. This function has multiple real-valued solutions at each point, so the ANN model needs to have the capacity to handle this in it's loss function. An MDN is a good way to handle the predictions of these multiple output values.

---

*There's a couple of other versions of this task, and this implementation owes much to the following:*

- [*David Ha - Mixture Density Networks with TensorFlow*](http://blog.otoro.net/2015/11/24/mixture-density-networks-with-tensorflow/)
- [*Mixture Density Networks in Edward*](http://edwardlib.org/tutorials/mixture-density-network)

In [ ]:
#!pip install git+git://github.com/cpmpercussion/keras-mdn-layer.git #egg=keras-mdn-layer
!pip install keras-mdn-layer
###!python3 -m pip install keras-mdn-layer

from tensorflow.compat.v1 import keras
###import keras
import keras_mdn_layer as mdn
#from context import * # imports the MDN layer
import numpy as np
import random
import matplotlib.pyplot as plt
#%matplotlib notebook

from sklearn.model_selection import train_test_split

## Generate Synthetic Data

Data generation

In [ ]:
## Generating some data:
NSAMPLE = 3000


# Fitted function y=f(x)
X = np.random.uniform(-4,4,NSAMPLE)
y = np.sin(X)*np.exp(0.01*X)+np.cos(X)
err = np.random.normal(0,0.3,NSAMPLE)
y = y + err

# Function x=f(y)
#y = np.random.uniform(-4,4,NSAMPLE)
#X = np.sin(y)*np.exp(0.01*y)+np.cos(y)
#err = np.random.normal(0,0.3,NSAMPLE)
#y = y + err

plt.figure(figsize=(8, 8))
plt.plot(X,y,'ro', alpha=0.3)
plt.show()


# Split into training and test sample
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=1)



## Build the MDN Model

Now we will construct the MDN model in Keras. This uses the `Sequential` model interface in Keras.

The `MDN` layer comes after one or more `Dense` layers. You need to define the output dimension and number of mixtures for the MDN like so: `MDN(output_dimension, number_mixtures)`.

For this problem, we only need an output dimension of 1 as we are predicting one value (y). Adding more mixtures adds a more parameters (model is more complex, takes longer to train), but might help make the solutions better. You can see from the training data that there are at maximum 5 different layers to predict in the curve, so setting `N_MIXES = 5` is a good place to start.

For MDNs, we have to use a special loss function that can handle the mixture parameters: the function has to take into account the number of output dimensions and mixtures.

In [ ]:
N_HIDDEN = 15
N_MIXES = 5

model = keras.Sequential()
model.add(keras.layers.Dense(N_HIDDEN, input_shape=(1,), activation='relu'))
model.add(keras.layers.Dense(N_HIDDEN, activation='relu'))
model.add(mdn.MDN(1, N_MIXES))

model.compile(loss=mdn.get_mixture_loss_func(1,N_MIXES), optimizer=keras.optimizers.Adam()) #, metrics=[mdn.get_mixture_mse_accuracy(1,N_MIXES)])
model.summary()

### Training the model

Now we train the model using Keras' normal `fit` command.

In [ ]:
history = model.fit(x=X_train, y=y_train, batch_size=128, epochs=100, validation_data=(X_test,y_test))

### Training and Validation Loss

It's interesting to see how the model trained. We can see that after a certain point training is rather slow.

For this problem a loss value around 1.5 produces quite good results.

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.show()

## Try out the MDN Model

Now we try out the model by making predictions at 3000 evenly spaced points on the x-axis.

Mixture models output lists of parameters, so we're going to sample from these parameters for each point on the x-axis, and also try plotting the parameters themselves so we can have some insight into what the model is learning!

In [ ]:
## Sample on some test data:
x_sample = np.float32(np.arange(min(X_test),max(X_test),0.01))
NTEST = x_sample.size
print("Testing:", NTEST, "samples.")
x_sample = x_sample.reshape(NTEST,1) # needs to be a matrix, not a vector

# Make predictions from the model
y_sample = model.predict(x_sample)
# y_sample contains parameters for distributions, not actual points on the graph.
# To find points on the graph, we need to sample from each distribution.

# Sample from the predicted distributions
y_samples = np.apply_along_axis(mdn.sample_from_output, 1, y_sample, 1, N_MIXES,temp=1.0)

# Split up the mixture parameters (for future fun)
mus = np.apply_along_axis((lambda a: a[:N_MIXES]),1, y_sample)
sigs = np.apply_along_axis((lambda a: a[N_MIXES:2*N_MIXES]),1, y_sample)
pis = np.apply_along_axis((lambda a: mdn.softmax(a[2*N_MIXES:])),1, y_sample)

In [ ]:
# Plot the samples
plt.figure(figsize=(8, 8))
plt.plot(X,y,'ro', x_sample, y_samples[:,0], 'bo',alpha=0.3)
plt.show()
# These look pretty good!

In [ ]:
# Plot the means - this gives us some insight into how the model learns to produce the mixtures.
plt.figure(figsize=(8, 8))
plt.plot(X_train,y_train,'ro', x_sample, mus,'bo',alpha=0.3)
plt.show()
# Cool!

In [ ]:
# Let's plot the variances and weightings of the means as well.
fig = plt.figure(figsize=(8, 8))
ax1 = fig.add_subplot(111)
# ax1.scatter(data[0], data[1], marker='o', c='b', s=data[2], label='the data')
ax1.scatter(X_train,y_train,marker='o', c='r', alpha=0.3)
for i in range(N_MIXES):
    ax1.scatter(x_sample, mus[:,i], marker='o', s=200*sigs[:,i]*pis[:,i],alpha=0.3)
plt.show()


---

#<font color='red'> **Exercise:**

Please use this generated data:


```python
# Function x=f(y)
y = np.random.uniform(-4,4,NSAMPLE)
X = np.sin(y)*np.exp(0.01*y)+np.cos(y)
err = np.random.normal(0,0.3,NSAMPLE)
y = y + err
```






# **Application of MDN**

_How do Mixture Density RNNs Predict the Future?<br> Kai O\. Ellefsen\, Charles P\. Martin\,Jim Torresen_

[https://arxiv\.org/pdf/1901\.07859\.pdf](https://arxiv.org/pdf/1901.07859.pdf)

__Problem: predicting the future in the computer game\.__

<img src='https://raw.githubusercontent.com/marcinwolter/NORCC-SUMMER_2022/main/images/DeepLearningLecture2020_617.png' width=350px>

<img src='https://raw.githubusercontent.com/marcinwolter/NORCC-SUMMER_2022/main/images/DeepLearningLecture2020_618.png' width=350px>

Model returns the series of predictions based on the previous frames. For each prediction it returns an error as well.

# **Could we use MDN also for classification?**


While MDNs are not standard classifiers (like Softmax classifiers), they can be adapted for classification.

### Why traditional classification doesn't use MDNs

In standard classification (e.g., MNIST digits), a single image of a "4" maps to one correct label (4). This is a one-to-one mapping. Standard neural networks with Softmax outputs handle this perfectly.

### Why you can use MDNs for classification?
An MDN can model the confidence of a prediction. If the model is uncertain (e.g., an image looks like both a 3 and an 8), the MDN will output a mixture with high variance or split weights.
In the  MNIST example: The MDN is essentially acting like an ensemble of n classifiers (since N_MIXES = n). It averages their "votes" (the means) to decide the final digit. This works, but it's stretching the original intent of the algorithm .

# Simple MNIST dense net with Mixture Density Network

Using on:
https://keras.io/examples/vision/mnist_convnet/

## Setup

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt


## Prepare the data

In [ ]:

def prepare_data(x_train, y_train, x_test, y_test,num_classes):
  #Select 10 classes
  N_CLASSES = num_classes

  indices = np.where(y_train < N_CLASSES) # if few classes needed
  indices = indices[0]
  np.random.shuffle(indices)
  x_train = x_train[indices]
  y_train = y_train[indices]

  indices = np.where(y_test < N_CLASSES) # if few  classes only
  indices = indices[0]
  np.random.shuffle(indices)
  x_test = x_test[indices]
  y_test = y_test[indices]

  # Scale images to the [0, 1] range
  x_train = x_train.astype("float32") / 255
  x_test = x_test.astype("float32") / 255
  # Make sure images have shape (28, 28, 1)
  x_train = np.expand_dims(x_train, -1)
  x_test = np.expand_dims(x_test, -1)
  print("x_train shape:", x_train.shape)
  print(x_train.shape[0], "train samples")
  print(x_test.shape[0], "test samples")


  # convert class vectors to binary class matrices
  y_train = keras.utils.to_categorical(y_train, num_classes)
  y_test = keras.utils.to_categorical(y_test, num_classes)

  return x_train, y_train, x_test, y_test

In [ ]:
# Model / data parameters
num_classes = 10
input_shape = (28, 28, 1)

# the data, split between train and test sets
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train, y_train, x_test, y_test = prepare_data(x_train, y_train, x_test, y_test,num_classes)

# **Install keras-mdn-layer**

In [ ]:
! pip install keras-mdn-layer
import keras_mdn_layer as mdn

Parameters for MDN

In [ ]:
N_HIDDEN = 128  # number of hidden units in the Dense layer
N_MIXES = 3  # number of mixture components
OUTPUT_DIMS = num_classes  # number of real-values predicted by each mixture component


## Build the model

In [ ]:
model = keras.Sequential()
model.add(keras.Input(shape=input_shape))
model.add(layers.Flatten())
model.add(layers.Dense(N_HIDDEN, activation="relu"))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(N_HIDDEN, activation="relu"))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(N_HIDDEN, activation="relu"))
model.add(layers.Dropout(0.2))
# Here comes the MDN layer
model.add(mdn.MDN(OUTPUT_DIMS, N_MIXES))


model.summary()

## Train the model

In [ ]:
batch_size = 128
epochs = 20

# This was for normal dense network
#model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.compile(loss=mdn.get_mixture_loss_func(OUTPUT_DIMS,N_MIXES), optimizer=keras.optimizers.Adam(learning_rate=5e-5))


history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(x_test, y_test)) #validation_split=0.1)

# **Plot loss**

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.show()

# **Model prediction with MDN**

In [ ]:
y_test_out = model.predict(x_test)

# y_samples - output from MDN, contains all information
y_samples = np.apply_along_axis(mdn.sample_from_output, 1, y_test_out, OUTPUT_DIMS, N_MIXES, temp=1.0)




In [ ]:

# Split up the mixture parameters (for future fun)
# means of Gaussians
mus = y_test_out[:,:N_MIXES*OUTPUT_DIMS]
# sigmas of Gaussians
sigs = y_test_out[:,N_MIXES*OUTPUT_DIMS:2*N_MIXES*OUTPUT_DIMS]
#pis = mdn.softmax(y_test_out[:,-N_MIXES:], t=1.0)


In [ ]:
# use the model to predict the labels of the test data



# Plot the prediction
fig = plt.figure(figsize=(8, 30))  # figure size in inches
fig.subplots_adjust(left=0, right=1, bottom=0, top=1, hspace=0.05, wspace=0.05)


# plot the digits: each image is 28x28 pixels
n_img=30
for i in range(n_img):
    ax = fig.add_subplot(n_img, 3, 3*i + 1, xticks=[], yticks=[])
    ax.imshow(x_test[i].reshape(28,28), cmap=plt.cm.binary, interpolation='nearest')

    # Reshape mus[i] and sigs[i] from (N_MIXES * OUTPUT_DIMS,) to (OUTPUT_DIMS, N_MIXES)

    current_mus_reshaped = mus[i].reshape(N_MIXES, OUTPUT_DIMS)
    current_sigs_reshaped = sigs[i].reshape(N_MIXES, OUTPUT_DIMS)

    # Calculate a representative mean and sigma for each class by averaging across mixture components
    # For a more robust approach, a weighted average using pi probabilities or considering variance of mixtures might be better
    mean_per_class = np.mean(current_mus_reshaped, axis=0)
    sigma_per_class = np.mean(current_sigs_reshaped, axis=0)

    ax = fig.add_subplot(n_img, 3, 3*i + 2, xticks=[0,1,2,3,4,5,6,7,8,9], yticks=[])
    xbar = np.linspace(1, OUTPUT_DIMS, num=OUTPUT_DIMS)
    ax.bar(xbar, mean_per_class, yerr=sigma_per_class, xerr=0.3)
    ax.axis('off')

    ax = fig.add_subplot(n_img, 3, 3*i + 3, xticks=[0,1,2,3,4,5,6,7,8,9], yticks=[])
    xbar = np.linspace(1, OUTPUT_DIMS, num=OUTPUT_DIMS)
    ax.bar(xbar,y_test[i],yerr=0.0,xerr=0.3)
    ax.axis('off')

